# Lesson 10 — A ladder of strikes is a distribution, one subtraction at a time

A threshold market quotes one point of a survival function. Quote several and the mass between two strikes is what their prices differ by, so a ladder of quotes is an implied probability distribution that nobody had to state.

**The rule.** `pmf(kᵢ, kᵢ₊₁] = S(kᵢ) − S(kᵢ₊₁)`

**When it holds.** Along any ladder of strikes on one underlying, where two adjacent strikes are both quoted.

**When it fails.** Reading a mean off the chart. The outermost bins are open — mass above the highest strike has no width and no midpoint — so a mean computed by pretending they sit at their bounds is a property of that convention rather than of the market. The moments here are conditional on the interior and say so.

| | |
|---|---|
| Lesson id | `distribution` |
| Pane it appears on | `lattice` (panes carry more than one lesson) |
| Code it is about | `modules/coherence/kernel/distribution.py`, `modules/coherence/kernel/moments.py` |
| Tests that go red if it stops being true | `tests/test_coherence_distribution.py` |
| Pane shipped | yes |

Every cell below runs against the real kernel. Nothing here is a re-implementation:
a number this notebook prints is the number the engine would produce for the same
input. The recorded Kalshi payloads come from `tests/fixtures/coherence/`.

In [ ]:
import json
import sys
from decimal import Decimal
from pathlib import Path

# This notebook lives in notebooks/coherence_lab/ and imports the kernel two
# levels up. Found by walking upward rather than by counting parents, so the
# notebook runs from its own directory or from Part2_Infrastructure.
HERE = Path.cwd().resolve()
ROOT = next((path for path in (HERE, *HERE.parents) if (path / "modules" / "coherence" / "kernel").is_dir()), None)
if ROOT is None:
    raise SystemExit(f"no coherence kernel above {HERE}: open this notebook from inside Part2_Infrastructure")
sys.path.insert(0, str(ROOT))

FIXTURES = ROOT / "tests" / "fixtures" / "coherence"


def fixture(name: str) -> dict:
    """One recorded Kalshi response, envelope and all, exactly as it was sent.

    These are captures, not mocks. Where a number below looks odd it is because
    the exchange quoted it, and `tools/capture_kalshi_fixtures.py` re-records
    them.
    """
    return json.loads((FIXTURES / f"{name}.json").read_text(encoding="utf-8"))


print(f"kernel root       {ROOT}")
print(f"recorded fixtures {FIXTURES.is_dir()}")

## 1. A ladder of strikes is a survival function, sampled

In [ ]:
from modules.coherence.kernel import distribution, moments
from modules.coherence.kernel.book import Book, Level
from modules.coherence.kernel.lattice import Component, Edge, Node

#: strike, YES bid, NO bid. A two-cent spread, so the mid is bid + 0.01.
RUNGS = (
    ("100000", "0.8100", "0.1700"),
    ("102000", "0.6000", "0.3800"),
    ("104000", "0.3700", "0.6100"),
    ("106000", "0.1600", "0.8200"),
    ("108000", "0.0400", "0.9400"),
)


def ladder(rungs):
    """A threshold family and the two-sided books that quote it."""
    nodes = [
        Node(f"T{strike}", "SYN", "KXBTCD", 0, "greater", Decimal(strike), None, ("synthetic",), f"above {strike}")
        for strike, _yes, _no in rungs
    ]
    books = {
        node.ticker: Book(
            ticker=node.ticker,
            yes_bids=(Level(Decimal(yes_bid), 20_000),),
            no_bids=(Level(Decimal(no_bid), 20_000),),
        )
        for node, (_strike, yes_bid, no_bid) in zip(nodes, rungs, strict=True)
    }
    # Monotonicity along the ladder, adjacent pairs only — the edges
    # `build_component` derives from the venue's own strike metadata.
    edges = [
        Edge(
            kind="implies", source=higher.ticker, target=lower.ticker, scope="same-event",
            because=(
                f"every outcome above {higher.floor_strike} is also above {lower.floor_strike}, "
                f"so P({higher.label}) cannot exceed P({lower.label})"
            ),
        )
        for lower, higher in zip(nodes, nodes[1:], strict=False)
    ]
    component = Component(
        component_id="SYN", event_ticker="SYN", series_ticker="KXBTCD",
        exchange_index=0, mutually_exclusive=False, nodes=nodes, edges=edges,
    )
    return component, books


family, books = ladder(RUNGS)
surface = distribution.build_surface(family, books)

print(f"  {surface.detail}, read on the {surface.basis} side")
print()
print("  strike     survival S(k)")
for probe in surface.probes:
    print(f"  {probe.strike}     {probe.survival}   ({probe.origin})")

## 2. Difference it, and the masses telescope to exactly one

In [ ]:
print("  interval                     mass = S(low) - S(high)")
for item in surface.bins:
    print(f"  {item.label:<26}   {item.mass}")
print()
total = sum((item.mass for item in surface.bins), Decimal(0))
print(f"  masses total {total}")
print(f"  exactly one, as Decimals: {total == Decimal(1)}")
print()
print("  It cannot come out otherwise, and the reason is worth writing down rather")
print("  than checking. The sum telescopes:")
print()
print("     (1 - S1) + (S1 - S2) + (S2 - S3) + ... + (S(n-1) - Sn) + Sn  =  1")
print()
print("  every interior survival value appearing once with each sign. So a pmf that")
print("  fails to total one is a defect in the reader, never a fact about the market —")
print("  and totalling the bins is therefore useless as a coherence test.")

## 3. The moments are conditional on the bounded interior

In [ ]:
interior = [(item.representative, item.mass) for item in surface.bins if item.representative is not None]
excluded = (surface.tail_mass_low or Decimal(0)) + (surface.tail_mass_high or Decimal(0))

print(f"  bounded interior bins : {len(interior)} of {len(surface.bins)}")
print(f"  tail below the lowest strike  : {surface.tail_mass_low}")
print(f"  tail above the highest strike : {surface.tail_mass_high}")
print(f"  excluded from every moment    : {excluded}")
print()
print(f"  mean              {surface.mean.quantize(Decimal('0.01'))}")
print(f"  variance          {surface.variance.quantize(Decimal('0.01'))}")
print(f"  skewness          {surface.skewness.quantize(Decimal('0.0001'))}")
print(f"  excess kurtosis   {surface.excess_kurtosis.quantize(Decimal('0.0001'))}")
print()
print(f"  {surface.moments_note}")
print()

# The same call, made directly, on the same points. `moments.central` knows
# nothing about strikes or books: it takes points on a line with weights.
mean, _var, _skew, _ex, note = moments.central(interior)
print(f"  moments.central on those {len(interior)} points gives mean {mean.quantize(Decimal('0.01'))}, the same")
print(f"  number, because it is the same call: {note}")
print()
print("  The top bin is 'above 108000'. It has mass and no width, so it has no")
print("  representative point, so it is left out rather than given an invented one. A")
print("  mean that pretended it sat at its floor would be a property of that convention.")

## 4. Break monotonicity and the bin goes below the axis

In [ ]:
# One rung repriced: 104000 now trades DEARER than 102000, which says an outcome
# above 104000 is likelier than one above 102000 — when every outcome in the
# first set is in the second.
BROKEN = (
    ("100000", "0.8100", "0.1700"),
    ("102000", "0.6000", "0.3800"),
    ("104000", "0.7100", "0.2700"),
    ("106000", "0.1600", "0.8200"),
    ("108000", "0.0400", "0.9400"),
)
broken_family, broken_books = ladder(BROKEN)
broken = distribution.build_surface(broken_family, broken_books)

for item in broken.bins:
    flag = "   NEGATIVE MASS" if item.is_negative else ""
    print(f"  {item.label:<26}   {item.mass}{flag}")
print()
print(f"  negative bins : {broken.negative_bins}")
print(f"  total mass    : {sum((item.mass for item in broken.bins), Decimal(0))}")
print()
print("  Still exactly one. The telescoping identity holds over a broken ladder just as")
print("  well, which is precisely why a total is not a test and a bar below the axis is.")

## 5. The same fault, priced

In [ ]:
from modules.coherence.kernel import closedform
from modules.coherence.kernel.constraints import rows_for
from modules.coherence.kernel.costs import FeeSchedule

rows = rows_for(broken_family, broken_books, families=("monotone",))
violated = [row for row in rows if row.violated]
print(f"  {len(rows)} monotone rows tested, {len(violated)} violated")
for row in violated:
    print(f"    slack {row.slack} on {row.executable_size_hundredths / 100} contracts")
print()
print(closedform.solve(broken_family, rows, FeeSchedule()).render_text())
print()
print("  The pmf SHOWS the fault as a bar below the axis. The constraint row PRICES it.")
print("  They are the same fault seen from two sides, and neither is derived from the")
print("  other — which is why the negative bin is worth rendering rather than repairing.")